<div dir="rtl" style="text-align:right">

# 🤖 مدل‌های زبانی بزرگ

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SharifiZarchi/IntroAI/blob/main/Session_10/LLM/LLM_Tutorial_FA.ipynb) [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FSharifiZarchi%2FIntroAI%2Fblob%2Fmain%2FSession_10%2FLLM%2FLLM_Tutorial_FA.ipynb)

جلسه پایانی. در جلسه ۹ یک لایه ترنسفورمر پیاده کردی؛ این‌جا روی هم می‌چینی‌اش، آموزشش می‌دهی و می‌بینی که یک مدل زبانی ظاهر می‌شود. بعد یک مدل واقعی را به کار می‌گیری، به آن دانش می‌دهی، به آن ابزار می‌دهی و دوره را تمام می‌کنی. مسیر:

۱. ‏GPT خودت را آموزش بده، از همان لایه‌ای که ساختی

۲. یک دستیار واقعی و روش کار با آن

۳. بازیابی: دادن دانشی که مدل ندارد

۴. عامل (agent): دادن ابزار و حلقه به مدل

۵. مقیاس، مراحل آموزش، محدودیت‌ها و جمع‌بندی دوره

پیش‌نیازها: لایه ترنسفورمر، ماسک علی و نمونه‌گیری با softmax (جلسه ۹)، و حلقه آموزش PyTorch (جلسه‌های ۷ و ۸).

**همه‌چیز این‌جا با حساب رایگان اجرا می‌شود.** بدون کلید API، بدون اشتراک پولی و بدون لایسنس دروازه‌دار. در Colab گزینه GPU را روشن کن (*Runtime، Change runtime type، T4 GPU*) تا بخش ۱ در حدود یک دقیقه آموزش ببیند؛ روی CPU حدود سه دقیقه طول می‌کشد. دستیار بخش‌های ۲ تا ۴ مدلی ۰.۵ میلیاردی است، حدود ۱ گیگابایت که یک بار دانلود می‌شود.

**روش اجرا:** روی هر سلول کلیک کن و `Shift + Enter` بزن، به ترتیب از بالا به پایین. هر بخش با تمرین تمام می‌شود.

(کد و مثال‌ها در هر دو نسخه فارسی و انگلیسی دقیقا یکسان و انگلیسی‌اند؛ فقط توضیح‌ها به دو زبان نوشته شده‌اند.)

</div>

<div dir="rtl" style="text-align:right">

---
# بخش ۱: ‏GPT خودت را آموزش بده

**مدل زبانی** به توکن بعدی، به شرط توکن‌های تاکنون، احتمال می‌دهد. کل تابع هدف همین است، و تولید متن فقط نمونه‌گیری مکرر از آن است: پیش‌بینی کن، یک توکن بکش، اضافه کن، تکرار کن.

داده آموزش هر متنی است و برچسب لازم ندارد، چون کاراکتر بعدی هر موقعیت خودش برچسب همان موقعیت است. از مجموعه نمایشنامه‌های شکسپیر (۱.۱ مگابایت) استفاده می‌کنیم و در سطح **کاراکتر** کار می‌کنیم تا واژگان ۶۵ نماد بماند و آموزش چند دقیقه بیشتر طول نکشد:

</div>

In [ ]:
import os
import time
import urllib.request
import torch
from torch import nn
import matplotlib.pyplot as plt
%matplotlib inline

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
if not os.path.exists("tinyshakespeare.txt"):
    urllib.request.urlretrieve(url, "tinyshakespeare.txt")
text = open("tinyshakespeare.txt", encoding="utf-8").read()

chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}

data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
split = int(0.9 * len(data))
train_data, val_data = data[:split], data[split:]

print("characters:", len(text), "| vocabulary:", vocab_size)
print("sample:", repr(text[:90]))

<div dir="rtl" style="text-align:right">

### ۱.۱ مدل
سه قطعه، همه از جلسه ۹. جدول **تعبیه توکن** شناسه هر کاراکتر را به بردار تبدیل می‌کند. **تعبیه مکان** موقعیت را اضافه می‌کند، چون توجه به‌تنهایی به ترتیب کور است. بعد چهار **لایه ترنسفورمر**، دقیقا همان کلاسی که جلسه قبل نوشتی، با این تفاوت که حالا ماسک علی روشن است تا هیچ موقعیتی آینده را نخواند:

</div>

In [ ]:
BLOCK, BATCH, D_MODEL, HEADS, LAYERS = 128, 64, 128, 4, 4

class TransformerLayer(nn.Module):
    def __init__(self, d_model, heads):
        super().__init__()
        self.attention = nn.MultiheadAttention(d_model, heads, batch_first=True)
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, 4 * d_model), nn.ReLU(), nn.Linear(4 * d_model, d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, mask):
        h = self.norm1(x)
        x = x + self.attention(h, h, h, attn_mask=mask, need_weights=False)[0]
        return x + self.feed_forward(self.norm2(x))


class TinyGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, D_MODEL)
        self.position_embedding = nn.Embedding(BLOCK, D_MODEL)
        self.layers = nn.ModuleList([TransformerLayer(D_MODEL, HEADS) for _ in range(LAYERS)])
        self.norm = nn.LayerNorm(D_MODEL)
        self.head = nn.Linear(D_MODEL, vocab_size)

    def forward(self, idx):
        T = idx.shape[1]
        mask = torch.triu(torch.ones(T, T, dtype=torch.bool, device=idx.device), diagonal=1)
        x = self.token_embedding(idx) + self.position_embedding(torch.arange(T, device=idx.device))
        for layer in self.layers:
            x = layer(x, mask)
        return self.head(self.norm(x))


device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)
model = TinyGPT().to(device)

print("device:", device)
print("parameters:", sum(p.numel() for p in model.parameters()))

<div dir="rtl" style="text-align:right">

‏۸۲۶٬۴۳۳ پارامتر، حدود شش مرتبه بزرگی کوچک‌تر از یک مدل مرزی، و معماری همان است.

### ۱.۲ بچ‌ها و نمونه‌گیری
هر نمونه آموزشی پنجره‌ای ۱۲۸ کاراکتری است و برچسبش همان پنجره است که یک واحد جابه‌جا شده، پس هر موقعیت جانشین خودش را پیش‌بینی می‌کند. تولید متن هر بار یک کاراکتر از توزیع خروجی مدل نمونه می‌گیرد، پس از تقسیم بر **دما** که تیزی توزیع را تعیین می‌کند:

</div>

In [ ]:
def get_batch(source):
    idx = torch.randint(len(source) - BLOCK - 1, (BATCH,))
    x = torch.stack([source[i:i + BLOCK] for i in idx])
    y = torch.stack([source[i + 1:i + BLOCK + 1] for i in idx])
    return x.to(device), y.to(device)


@torch.no_grad()
def generate(model, seed="ROMEO:", n=200, temperature=0.8):
    model.eval()
    idx = torch.tensor([[stoi.get(c, 0) for c in seed]], device=device)
    for _ in range(n):
        logits = model(idx[:, -BLOCK:])[:, -1, :] / temperature
        probabilities = torch.softmax(logits, dim=-1)
        idx = torch.cat([idx, torch.multinomial(probabilities, 1)], dim=1)
    model.train()
    return "".join(itos[i] for i in idx[0].tolist())


x, y = get_batch(train_data)
print("input batch:", tuple(x.shape), "| target batch:", tuple(y.shape))
print("before training:", repr(generate(model, n=60)))

<div dir="rtl" style="text-align:right">

مدل آموزش‌ندیده نویز تولید می‌کند، همان‌طور که باید: وزن‌هایش تصادفی‌اند.

### ۱.۳ آموزش
حلقه همان حلقه جلسه ۷ است، بدون تغییر: گذر رو به جلو، هزینه، گذر رو به عقب، گام. هر چند صد گام یک نمونه چاپ می‌کنیم تا یادگیری در همان لحظه دیده شود:

</div>

In [ ]:
STEPS = 2000 if device == "cuda" else 800          # keeps the runtime near a few minutes
CHECKPOINTS = [0, 200, STEPS // 2, STEPS]

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)
history = []

start = time.time()
for step in range(STEPS + 1):
    if step in CHECKPOINTS:
        with torch.no_grad():
            vx, vy = get_batch(val_data)
            val_loss = loss_fn(model(vx).reshape(-1, vocab_size), vy.reshape(-1)).item()
        print(f"step {step:5d} | val loss {val_loss:.3f} | {time.time() - start:.0f}s")
        print(generate(model, n=110))
        print("-" * 70)

    x, y = get_batch(train_data)
    loss = loss_fn(model(x).reshape(-1, vocab_size), y.reshape(-1))
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    history.append(loss.item())

print(f"trained in {time.time() - start:.0f}s on {device}")

<div dir="rtl" style="text-align:right">

چهار نمونه را به ترتیب بخوان، چون همین دنباله کل حرف این جلسه است.

در گام ۰ خروجی کاراکترهای تصادفی است. تا گام ۲۰۰ مدل شکل زبان انگلیسی را یاد گرفته: خوشه‌های حروف قابل تلفظ، فاصله‌ها در جای باورپذیر، و قالب نمایشنامه یعنی نام گوینده و بعد دونقطه. در ایست میانی، واژه‌ها بیشتر واقعی‌اند و سطرها آهنگ شعر دارند. در پایان، دیالوگ می‌نویسد با نام درست شخصیت‌ها و نقطه‌گذاری و ساختار صحنه؛ از مدلی با کمتر از یک میلیون پارامتر که چند دقیقه روی یک مگابایت متن آموزش دیده.

هیچ‌کس به آن نگفت واژه چیست، یا نام‌ها کدام‌اند، یا این‌که بعد از نام گوینده دونقطه می‌آید. همه‌اش از پیش‌بینی کاراکتر بعدی بیرون آمد.

منحنی هزینه، هموارشده روی ۵۰ گام:

</div>

In [ ]:
smoothed = torch.tensor(history).unfold(0, 50, 25).mean(dim=1)

plt.figure(figsize=(8, 3.5))
plt.plot(torch.arange(len(smoothed)) * 25, smoothed)
plt.xlabel("training step"); plt.ylabel("training loss")
plt.title("the tiny GPT learning to write")
plt.grid(alpha=0.3)
plt.show()

<div dir="rtl" style="text-align:right">

### ۱.۴ دما
نمونه‌گیری پیش از softmax لاجیت‌ها را بر دما تقسیم می‌کند. دمای پایین توزیع را تیز می‌کند و مدل امن‌ترین الگوهایش را تکرار می‌کند؛ دمای بالا صافش می‌کند و مدل ابداع می‌کند:

</div>

In [ ]:
for temperature in (0.2, 0.8, 1.5):
    print(f"=== temperature {temperature} ===")
    print(generate(model, n=140, temperature=temperature))
    print()

<div dir="rtl" style="text-align:right">

در ۰.۲ متن تکراری و محافظه‌کار است. در ۰.۸ متنوع اما منسجم. در ۱.۵ واژه‌های ساختگی و نام‌های شکسته تولید می‌کند.

همین اثر را در خود اعداد ببین. یک موقعیت واقعا نامطمئن برمی‌داریم، کاراکتر بعد از `"We are all "`، و توزیعش را در هر دما دوباره رسم می‌کنیم:

</div>

In [ ]:
prompt = "First Citizen:\nWe are all "
with torch.no_grad():
    logits = model(torch.tensor([[stoi[c] for c in prompt]], device=device))[0, -1].cpu()

top = logits.topk(10).indices
width = 0.27

plt.figure(figsize=(9, 3.5))
for offset, temperature in zip((-width, 0, width), (0.2, 0.8, 1.5)):
    probabilities = torch.softmax(logits / temperature, dim=-1)[top]
    plt.bar(torch.arange(10) + offset, probabilities, width, label=f"T = {temperature}")
plt.xticks(range(10), [repr(itos[int(i)]) for i in top])
plt.ylabel("probability"); plt.title('next character after "We are all "')
plt.legend(); plt.grid(axis="y", alpha=0.3)
plt.show()

<div dir="rtl" style="text-align:right">

یک مجموعه لاجیت، سه توزیع متفاوت. در دمای ۰.۲ تقریبا تمام جرم احتمال روی محتمل‌ترین کاراکتر جمع می‌شود، پس نمونه‌گیری عملا قطعی است و مدل امن‌ترین چیزش را تکرار می‌کند. در دمای ۱.۵ میله‌ها هم‌تراز می‌شوند و کاراکترهایی که مدل کم‌احتمال می‌داند شروع به بردن قرعه می‌کنند، و واژه‌های ساختگی از همین‌جا می‌آیند.

دقت کن که ترتیب رتبه‌ها هرگز عوض نمی‌شود، فقط فاصله میله‌ها. دما به مدل ایده تازه نمی‌دهد، بلکه تعیین می‌کند نمونه‌گیر چقدر حاضر است سراغ ایده‌هایی برود که مدل خودش پایین رتبه‌بندی کرده. هر رابط گفت‌وگویی همین پیچ را در اختیارت می‌گذارد و حالا دقیقا می‌دانی با اعداد چه می‌کند.

</div>

<div dir="rtl" style="text-align:right">

### ✏️ تمرین ۱
داخل پیش‌بینی را ببین. برای پرامپت `"ROMEO:"` توزیع خروجی مدل برای کاراکتر بعدی را بگیر و ده کاراکتر محتمل‌تر را با احتمالشان چاپ کن. مدل بعد از نام گوینده انتظار چه کاراکترهایی دارد؟

</div>

In [ ]:
# ✏️ top-10 next characters



<div dir="rtl" style="text-align:right">

<details>
<summary>💡 راه‌حل (برای بازشدن کلیک کن)</summary>

```python
context = torch.tensor([[stoi[c] for c in "ROMEO:"]], device=device)
with torch.no_grad():
    probs = torch.softmax(model(context)[0, -1], dim=-1)
top = probs.topk(10)
for p, i in zip(top.values, top.indices):
    print(f"{itos[int(i)]!r:6s} {p.item():.3f}")
```

کاراکتر خط جدید غالب است و بعد فاصله و حروف پربسامد، چون در متن آموزش تقریبا همیشه بعد از نام گوینده خط عوض می‌شود. مدل قالب نمایشنامه را یاد گرفته، نه فقط واژه‌هایش را.

</details>

</div>

<div dir="rtl" style="text-align:right">

### ✏️ تمرین ۲
از سه بذر متفاوت تولید کن، مثلا `"JULIET:"` و `"To be, or"` و `"The king"`. مدل در نقش می‌ماند؟ اگر بذر کاراکتری داشته باشد که هرگز ندیده، مثل رقم، چه می‌شود؟

</div>

In [ ]:
# ✏️ your seeds



<div dir="rtl" style="text-align:right">

<details>
<summary>💡 راه‌حل (برای بازشدن کلیک کن)</summary>

```python
for seed in ["JULIET:", "To be, or", "The king"]:
    print(f"=== {seed!r}")
    print(generate(model, n=120))
    print()
```

مدل در همان لحن بذر ادامه می‌دهد، چون بذر همان بافتی است که توجه می‌خواند. کاراکترهای بیرون از واژگان ۶۵ نمادی اصلا قابل کدگذاری نیستند و `stoi.get(c, 0)` بی‌صدا آن‌ها را به کاراکتر اول نگاشت می‌کند؛ این همان مشکل خارج از واژگان است که توکن‌سازهای زیرواژه‌ای در مدل‌های واقعی حلش می‌کنند.

</details>

</div>

<div dir="rtl" style="text-align:right">

### ✏️ تمرین ۳
طول بافت ابرپارامتری با اثر دیدنی است. مدل دومی با `BLOCK = 16` به‌جای ۱۲۸ و با همان تعداد گام آموزش بده و نمونه‌هایش را مقایسه کن. کدام نوع ساختار زودتر از بین می‌رود؟

</div>

In [ ]:
# ✏️ shorter context



<div dir="rtl" style="text-align:right">

<details>
<summary>💡 راه‌حل (برای بازشدن کلیک کن)</summary>

```python
BLOCK_BACKUP = BLOCK
BLOCK = 16
torch.manual_seed(42)
short_model = TinyGPT().to(device)
short_opt = torch.optim.AdamW(short_model.parameters(), lr=3e-3)
for step in range(STEPS):
    x, y = get_batch(train_data)
    loss = loss_fn(short_model(x).reshape(-1, vocab_size), y.reshape(-1))
    short_opt.zero_grad(); loss.backward(); short_opt.step()
print(generate(short_model, n=140))
BLOCK = BLOCK_BACKUP
```

واژه‌ها زنده می‌مانند، چون املا فقط چند کاراکتر بافت می‌خواهد، اما ساختار دوربرد از بین می‌رود: نام گوینده جای اشتباه می‌آید و سطرها به هم ربط ندارند. طول بافت انسجام می‌خرد، و برای همین مدل‌های مرزی این‌قدر برای افزایشش می‌جنگند، در برابر همان هزینه مربعی که در جلسه ۹ اندازه گرفتی.

</details>

</div>

<div dir="rtl" style="text-align:right">

### ✏️ تمرین ۴
نمونه‌گیری در برابر رمزگشایی حریصانه. تابع `generate` را طوری تغییر بده که همیشه محتمل‌ترین کاراکتر را بردارد (`probabilities.argmax()`) و خروجی را با دمای ۰.۸ مقایسه کن. چرا رمزگشایی حریصانه برای متن خلاقانه انتخاب بدی است؟

</div>

In [ ]:
# ✏️ greedy decoding



<div dir="rtl" style="text-align:right">

<details>
<summary>💡 راه‌حل (برای بازشدن کلیک کن)</summary>

```python
@torch.no_grad()
def generate_greedy(model, seed="ROMEO:", n=140):
    model.eval()
    idx = torch.tensor([[stoi.get(c, 0) for c in seed]], device=device)
    for _ in range(n):
        logits = model(idx[:, -BLOCK:])[:, -1, :]
        idx = torch.cat([idx, logits.argmax(dim=-1, keepdim=True)], dim=1)
    model.train()
    return "".join(itos[i] for i in idx[0].tolist())

print(generate_greedy(model))
```

رمزگشایی حریصانه در حلقه می‌افتد و یک عبارت را تکرار می‌کند، چون محتمل‌ترین کاراکتر بعدی در بافتی تکراری، ادامه همان تکرار است. نمونه‌گیری است که متن تولیدشده را زنده نگه می‌دارد و دما پیچ بین این دو رفتار است.

</details>

</div>

<div dir="rtl" style="text-align:right">

---
# بخش ۲: یک دستیار واقعی

مدل تو ۰.۸ میلیون پارامتر دارد و یک مگابایت خوانده. دستیار صنعتی میلیاردها پارامتر دارد و ترابایت‌ها خوانده، اما تابع هدف و معماری همان‌هایی است که همین الان به کار بردی.

مدل **Qwen2.5-0.5B-Instruct** را بار می‌کنیم: ۴۹۴ میلیون پارامتر، با لایسنس Apache-2.0، بدون نیاز به حساب یا توکن، حدود ۱ گیگابایت که یک بار دانلود می‌شود. آن‌قدر کوچک هست که در نوت‌بوک رایگان جا شود و از قبل برای پیروی از دستور آموزش دیده است:

</div>

In [ ]:
import os
os.environ["USE_TF"] = "0"

from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
llm = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device).eval()
if device == "cuda":
    llm = llm.half()                        # half precision is faster on a GPU

llm.generation_config.top_p = None          # we control sampling with temperature alone,
llm.generation_config.top_k = None          # so clear the packaged defaults and keep the
llm.generation_config.temperature = None    # output free of configuration warnings

print("parameters:", sum(p.numel() for p in llm.parameters()))
print("vocabulary:", tokenizer.vocab_size, "subword tokens")

<div dir="rtl" style="text-align:right">

### ۲.۱ قالب گفت‌وگو
مدل زبانی پایه فقط متن را ادامه می‌دهد. دستیار وقتی ساخته می‌شود که گفت‌وگو را در قالبی ثابت با نشانگر نقش بپیچیم، قالبی که مدل برای پیروی از آن آموزش دیده. همین قالب است که پیام سیستمی را ممکن می‌کند:

</div>

In [ ]:
messages = [{"role": "system", "content": "You are a helpful teacher."},
            {"role": "user", "content": "Explain gradient descent in one sentence."}]

print(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

<div dir="rtl" style="text-align:right">

توکن‌های ویژه مشخص می‌کنند هر نقش کجا شروع و کجا تمام می‌شود، و نشانگر آخر به مدل می‌گوید حالا نوبت دستیار است. تمام کاری که یک رابط گفت‌وگو می‌کند ساختن همین رشته است.

### ۲.۲ حرف‌زدن با آن
تابع کمکی زیر دقیقا همین کار را می‌کند و نه بیشتر: رشته را می‌سازد، به شناسه توکن تبدیلش می‌کند، تولید می‌کند، و فقط بخش تازه‌تولیدشده را رمزگشایی می‌کند:

</div>

In [ ]:
def chat(user_message, system="You are a helpful assistant. Answer in one short sentence.",
         max_new_tokens=60, temperature=0.7, do_sample=True):
    messages = [{"role": "system", "content": system},
                {"role": "user", "content": user_message}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    settings = {"temperature": temperature} if do_sample else {}   # temperature only applies when sampling
    with torch.no_grad():
        out = llm.generate(ids, max_new_tokens=max_new_tokens, do_sample=do_sample,
                           pad_token_id=tokenizer.eos_token_id, **settings)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()


for question in ["What is the capital of France?",
                 "Explain gradient descent in one sentence."]:
    print("Q:", question)
    print("A:", chat(question), "\n")

<div dir="rtl" style="text-align:right">

### ۲.۳ پیام سیستمی همه‌چیز را عوض می‌کند
یک پرسش واحد، سه دستور متفاوت درباره این‌که مدل کیست:

</div>

In [ ]:
question = "Why is the sky blue?"

for system in ["You are a physics professor. Answer in one technical sentence.",
               "You are talking to a five year old. Answer in one simple sentence.",
               "You are a poet. Answer in one line of verse."]:
    print(system)
    print("  ->", chat(question, system=system), "\n")

<div dir="rtl" style="text-align:right">

### ۲.۴ یادگیری داخل پرامپت
دو مثال از یک قالب ساختگی به مدل بده تا برای ورودی سوم همان الگو را دنبال کند، بدون هیچ آموزشی و بدون هیچ به‌روزرسانی وزنی. به این **یادگیری در بافت** می‌گویند و وقتی مدل‌ها به اندازه کافی بزرگ شدند، خودبه‌خود ظاهر شد:

</div>

In [ ]:
few_shot = '''Convert the sentence into a command.

Sentence: I would like the light to be brighter.
Command: increase_brightness()

Sentence: It is too loud in here.
Command: decrease_volume()

Sentence: The room is too cold.
Command:'''

print(chat(few_shot, system="You complete patterns exactly.", max_new_tokens=15, do_sample=False))

<div dir="rtl" style="text-align:right">

### ۲.۵ کجا شکست می‌خورد
دو شکست، هر دو آموزنده. اول حساب:

</div>

In [ ]:
print("model:", chat("What is 4321 * 1234? Reply with only the number.", max_new_tokens=30))
print("truth:", 4321 * 1234)

<div dir="rtl" style="text-align:right">

مدل عددی با اطمینان تولید می‌کند که غلط است، چون ضرب الگویی نیست که پیش‌بینی توکن بعدی به‌طور قابل‌اتکا یاد بگیرد. بخش ۴ این را با یک ابزار حل می‌کند.

دوم، فارسی. همان پرسش، یک بار انگلیسی و یک بار فارسی:

</div>

In [ ]:
print("EN:", chat("What is artificial intelligence? Answer in one sentence."), "\n")
print("FA:", chat("هوش مصنوعی چیست؟ در یک جمله توضیح بده."))

<div dir="rtl" style="text-align:right">

پاسخ انگلیسی روان است و پاسخ فارسی به‌شدت افت می‌کند و گاهی به واژه‌های شکسته می‌رسد. دلیلش داده است: این مدل در پیش‌آموزش انگلیسی بسیار بیشتری از فارسی دیده، و مدل ۰.۵ میلیاردی ظرفیت اضافه‌ای برای یک زبان کم‌منبع ندارد. مدل‌های چندزبانه بزرگ‌تر بسیار بهتر عمل می‌کنند، اما این شکاف در هر مقیاسی واقعی است و بستنش برای فارسی، داده‌ی فارسی و محک فارسی می‌خواهد. این یک میدان باز است و یکی از مفیدترین کارهایی است که فارغ‌التحصیل این دوره می‌تواند انجام دهد.

</div>

<div dir="rtl" style="text-align:right">

### ✏️ تمرین ۵
پیام سیستمی‌ای بنویس که مدل به هر پرسشی با یک پرسش دیگر جواب بدهد، به سبک معلم سقراطی، و روی دو پرسش امتحانش کن.

</div>

In [ ]:
# ✏️ your system prompt



<div dir="rtl" style="text-align:right">

<details>
<summary>💡 راه‌حل (برای بازشدن کلیک کن)</summary>

```python
socratic = "You are a Socratic teacher. Never answer directly. Reply with one guiding question."
for q in ["What is overfitting?", "Why do we split data into train and test?"]:
    print("Q:", q)
    print("A:", chat(q, system=socratic), "\n")
```

مدل کوچک دستور را ناقص رعایت می‌کند و گاهی باز هم جواب مستقیم می‌دهد. پیروی از دستور توانایی‌ای است که با مقیاس رشد می‌کند، و یکی از دلایلی است که دستیارهای بزرگ حرف‌شنوتر به نظر می‌رسند.

</details>

</div>

<div dir="rtl" style="text-align:right">

### ✏️ تمرین ۶
اندازه بگیر که دما با یک دستیار واقعی چه می‌کند. یک پرسش را پنج بار با `temperature=0.2` و پنج بار با `temperature=1.5` بپرس و تعداد پاسخ‌های متمایز را مقایسه کن.

</div>

In [ ]:
# ✏️ diversity versus temperature



<div dir="rtl" style="text-align:right">

<details>
<summary>💡 راه‌حل (برای بازشدن کلیک کن)</summary>

```python
for temperature in (0.2, 1.5):
    answers = {chat("Give me a name for a pet cat.", max_new_tokens=12, temperature=temperature)
               for _ in range(5)}
    print(f"temperature {temperature}: {len(answers)} distinct answers")
    for a in answers:
        print("   ", a)
```

دمای پایین تقریبا هر بار همان پاسخ را می‌دهد و دمای بالا پنج پاسخ متفاوت و گاهی بی‌ربط. همان بده‌بستانی است که روی مدل خودت در بخش ۱ دیدی، در مقیاسی متفاوت.

</details>

</div>

<div dir="rtl" style="text-align:right">

### ✏️ تمرین ۷
یادگیری در بافت را با قالبی از ابداع خودت امتحان کن، مثلا تبدیل یک جمله به شیء JSON با دو فیلد. دو مثال، بعد ورودی سوم. مدل قالب را دقیقا نگه می‌دارد؟

</div>

In [ ]:
# ✏️ your few-shot prompt



<div dir="rtl" style="text-align:right">

<details>
<summary>💡 راه‌حل (برای بازشدن کلیک کن)</summary>

```python
prompt = '''Extract the fields.

Text: The meeting is on Monday at 9.
JSON: {"day": "Monday", "time": "9"}

Text: Lunch is on Friday at 12.
JSON: {"day": "Friday", "time": "12"}

Text: The exam is on Sunday at 8.
JSON:'''
print(chat(prompt, system="You complete patterns exactly.", max_new_tokens=25, do_sample=False))
```

مدل معمولا ساختار را درست بازتولید می‌کند، چون الگو صریح در پنجره بافت است. سازوکار پشت بیشتر مهندسی پرامپت عملی همین است: نشان‌دادن از توصیف‌کردن بهتر جواب می‌دهد.

</details>

</div>

<div dir="rtl" style="text-align:right">

---
# بخش ۳: بازیابی، دادن دانش به مدل

مدل فقط چیزی را می‌داند که وزن‌هایش در آموزش جذب کرده‌اند. از چیزی بیرون از آن بپرس، نمی‌گوید «نمی‌دانم»، بلکه حدسی روان تولید می‌کند، چون ادامه روان دقیقا همان چیزی است که برایش آموزش دیده.

راه‌حل استاندارد **تولید تقویت‌شده با بازیابی (RAG)** است: واقعیت‌ها را در مجموعه‌ای از سندها نگه دار، مرتبط‌ترین را پیدا کن و در پرامپت بگذار. این هم مجموعه کوچکی درباره همین دوره که هیچ مدلی هرگز نخوانده است:

</div>

In [ ]:
from transformers import AutoModel, logging as hf_logging

hf_logging.set_verbosity_error()      # BERT here is used without its pretraining head,
                                      # so silence the report about those unused weights

documents = [
    "The IntroAI course has ten sessions and is taught by Ali Sharifi Zarchi on YouTube.",
    "Session 8 of the IntroAI course covers computer vision: convolution, CNNs and transfer learning.",
    "Session 9 of the IntroAI course covers word embeddings, the transformer layer, and a Kaggle competition on disaster tweets.",
    "Every IntroAI notebook is published in both Persian and English in the SharifiZarchi repository on GitHub.",
    "Gradient descent is the connecting thread of the IntroAI course, from linear regression to language models.",
]

encoder_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
encoder = AutoModel.from_pretrained("bert-base-uncased").to(device).eval()

def embed(texts):
    batch = encoder_tokenizer(texts, padding=True, truncation=True, max_length=64,
                              return_tensors="pt").to(device)
    with torch.no_grad():
        hidden = encoder(**batch).last_hidden_state
    mask = batch["attention_mask"].unsqueeze(-1)
    vectors = (hidden * mask).sum(dim=1) / mask.sum(dim=1)
    return (vectors / vectors.norm(dim=1, keepdim=True)).cpu()

document_vectors = embed(documents)
print("document vectors:", tuple(document_vectors.shape))

<div dir="rtl" style="text-align:right">

بازیابی همان شباهت کسینوسی جلسه ۹ است و نه بیشتر: پرسش را بردار کن، با همه سندها مقایسه کن، بهترین را بردار. بعد پاسخ را با و بدون آن بافت مقایسه کن:

</div>

In [ ]:
def retrieve(question):
    scores = document_vectors @ embed([question])[0]
    best = int(scores.argmax())
    return documents[best], float(scores[best])


for question in ["Which session of the IntroAI course covers computer vision?",
                 "In which two languages are the IntroAI notebooks published?"]:
    passage, score = retrieve(question)
    print("Q:", question)
    print("  without context:", chat(question, max_new_tokens=40, do_sample=False))
    print(f"  retrieved (score {score:.2f}): {passage}")
    print("  with context   :", chat(f"Context: {passage}\n\nQuestion: {question}",
                                     max_new_tokens=40, do_sample=False), "\n")

<div dir="rtl" style="text-align:right">

بدون بافت مدل ابداع می‌کند: بینایی ماشین را در جلسه اول می‌گذارد و ادعا می‌کند نوت‌بوک‌ها به انگلیسی و آلمانی‌اند. با یک جمله بازیابی‌شده در پرامپت، هر دو پاسخ درست می‌شوند.

هیچ چیزی در مدل عوض نشد. وزن‌ها همان‌اند و فقط بافت فرق دارد. برای همین بازیابی اولین چیزی است که به تقریبا هر سیستم صنعتی مبتنی بر مدل زبانی اضافه می‌شود، و برای همین یک نمایه جست‌وجو به‌علاوه یک مدل کوچک اغلب از یک مدل بزرگ تنها بهتر است.

</div>

<div dir="rtl" style="text-align:right">

### ✏️ تمرین ۸
سه سند از خودت درباره موضوعی که مدل نمی‌تواند بداند اضافه کن، مثلا شهرت، دانشکده‌ات یا پروژه شخصی‌ات، و پرسشی بپرس که فقط سندهای تو جوابش را دارند.

</div>

In [ ]:
# ✏️ your documents and question



<div dir="rtl" style="text-align:right">

<details>
<summary>💡 راه‌حل (برای بازشدن کلیک کن)</summary>

```python
my_documents = documents + [
    "The IntroAI final session builds a tiny GPT, an assistant, a retrieval system and an agent.",
    "The IntroAI Kaggle notebook reaches an F1 score of about 0.82 with a fine-tuned transformer.",
]
document_vectors = embed(my_documents)
documents = my_documents

question = "What F1 score does the IntroAI Kaggle notebook reach?"
passage, score = retrieve(question)
print("without:", chat(question, max_new_tokens=40, do_sample=False))
print("with   :", chat(f"Context: {passage}\n\nQuestion: {question}", max_new_tokens=40, do_sample=False))
```

الگو برقرار است: بازیابی یک ماشین حدس‌زننده را به سیستمی تبدیل می‌کند که از شواهدی که در اختیار توست جواب می‌دهد. باقی یک سیستم RAG صنعتی، مهندسی حول همین هسته است: تکه‌کردن سندهای بلند، نمایه‌کردن میلیون‌ها بردار، و برگرداندن چند قطعه به‌جای یکی.

</details>

</div>

<div dir="rtl" style="text-align:right">

### ✏️ تمرین ۹
بشکنش. پرسشی بپرس که جوابش در هیچ سندی نیست و امتیاز بازیابی و پاسخ را وارسی کن. سیستم خوب وقتی بهترین امتیاز پایین است باید چه کند؟

</div>

In [ ]:
# ✏️ an out-of-scope question



<div dir="rtl" style="text-align:right">

<details>
<summary>💡 راه‌حل (برای بازشدن کلیک کن)</summary>

```python
question = "What is the price of a train ticket from Tehran to Isfahan?"
passage, score = retrieve(question)
print(f"best score {score:.2f}: {passage}")
print("answer:", chat(f"Context: {passage}\n\nQuestion: {question}", max_new_tokens=40, do_sample=False))
```

بازیابی همیشه بهترین تطبیقش را برمی‌گرداند، حتی وقتی هیچ‌چیز مرتبط نیست، و مدل بعد از روی قطعه‌ای بی‌ربط جواب می‌دهد. سیستم خوب آستانه‌ای برای شباهت می‌گذارد و وقتی چیزی از آن عبور نکرد می‌گوید نمی‌داند. بازیابی توهم را کم می‌کند، حذفش نمی‌کند.

</details>

</div>

<div dir="rtl" style="text-align:right">

---
# بخش ۴: عامل، دادن ابزار به مدل

بخش ۲ نشان داد مدل در حساب شکست می‌خورد. هرگز در آن قابل‌اتکا نخواهد شد و لازم هم نیست: ماشین‌حساب یک خط پایتون است. **عامل (agent)** یعنی یک مدل زبانی در حلقه‌ای با ابزارهایی که می‌تواند صدا بزند.

حلقه چهار قدم دارد. مدل پرسش و توصیف ابزارها را می‌گیرد؛ یا فراخوانی ابزار می‌نویسد یا پاسخ نهایی؛ اگر ابزار خواسته بود، اجرایش می‌کنیم و نتیجه را به آن برمی‌گردانیم؛ تا رسیدن به پاسخ یا سقف گام‌ها تکرار می‌شود.

توصیف ابزارها در پیام سیستمی است، و آن مثال آخر مهم‌تر از چیزی است که به نظر می‌رسد:

</div>

In [ ]:
import re

TOOL_PROMPT = '''You are an assistant that can use tools.

Available tools:
calculator(expression) - evaluates a Python arithmetic expression

To use a tool, reply with exactly one line:
ACTION: calculator("EXPRESSION")

When you know the answer, reply with exactly one line:
ANSWER: your answer

Example:
User: What is 25 * 4?
ACTION: calculator("25 * 4")
Observation: 100
ANSWER: 100'''


def calculator(expression):
    return eval(expression, {"__builtins__": {}}, {})       # arithmetic only, no names available


def step(messages, max_new_tokens=40):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    with torch.no_grad():
        out = llm.generate(ids, max_new_tokens=max_new_tokens, do_sample=False,
                           pad_token_id=tokenizer.eos_token_id)
    reply = tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()
    return reply.split(chr(10))[0].strip()                  # keep only the first line

print("agent parts ready ✅")

<div dir="rtl" style="text-align:right">

نگه‌داشتن فقط خط اول جزئیات نیست، همان چیزی است که حلقه را کار می‌اندازد. مدل به حال خود رها شود، فراخوانی ابزار *و* یک مشاهده ساختگی *و* یک پاسخ را در یک نفس می‌نویسد، پس هرگز خروجی واقعی ابزار را نمی‌بیند. بریدن در خط اول مجبورش می‌کند بایستد و منتظر بماند. چارچوب‌های واقعی عامل دقیقا همین کار را با دنباله‌های توقف می‌کنند.

حالا حلقه:

</div>

In [ ]:
def run_agent(question, max_steps=4, verbose=True):
    messages = [{"role": "system", "content": TOOL_PROMPT},
                {"role": "user", "content": question}]

    for _ in range(max_steps):
        reply = step(messages)
        if verbose:
            print("  model:", reply)

        call = re.search(r'ACTION:\s*calculator\("([^"]+)"\)', reply)
        if not call:
            answer = re.search(r"ANSWER:\s*(.+)", reply)
            return answer.group(1).strip() if answer else reply

        try:
            observation = calculator(call.group(1))
        except Exception as error:
            observation = f"error: {error}"
        if verbose:
            print("  tool :", observation)

        messages.append({"role": "assistant", "content": reply})
        messages.append({"role": "user", "content": f"Observation: {observation}"})

    return None


print("Q: What is 4321 * 1234?")
answer = run_agent("What is 4321 * 1234?")
print("FINAL:", answer)
print("truth:", 4321 * 1234)

<div dir="rtl" style="text-align:right">

رد اجرا کل سازوکار را نشان می‌دهد: مدل تشخیص می‌دهد که به حساب نیاز دارد، فراخوانی ابزار می‌نویسد، ابزار حاصل‌ضرب دقیق را برمی‌گرداند و مدل آن را اعلام می‌کند. همان مدلی که در بخش ۲ غلط حدس زد، حالا درست جواب می‌دهد، چون دست از حساب‌کردن خودش برداشت.

یک مسئله کلامی هم به همین شکل کار می‌کند، با این تفاوت که مدل خودش انتخاب می‌کند چه چیزی را حساب کند:

</div>

In [ ]:
print("Q: A book costs 17 dollars. How much do 23 books cost?")
print("FINAL:", run_agent("A book costs 17 dollars. How much do 23 books cost?"))

<div dir="rtl" style="text-align:right">

عامل یعنی همین: برنامه‌ریزی، فراخوانی، مشاهده، پاسخ. سیستم‌های صنعتی ابزار بیشتری اضافه می‌کنند (جست‌وجوی وب، اجرای کد، پایگاه داده)، قالب بهتری می‌گذارند (رابط بومی فراخوانی ابزار به‌جای عبارت باقاعده ما) و مدل‌های بزرگ‌تری می‌گذارند که چند قدم جلوتر را برنامه‌ریزی می‌کنند. شکل حلقه همان می‌ماند.

</div>

<div dir="rtl" style="text-align:right">

### ✏️ تمرین ۱۰
شکستش را تماشا کن. عامل را روی `"What is 15% of 240?"` اجرا کن و رد اجرا را بخوان. دقیقا چه چیزی غلط پیش می‌رود و شکست در برنامه‌ریزی است یا در قالب‌بندی؟

</div>

In [ ]:
# ✏️ the percentage question



<div dir="rtl" style="text-align:right">

<details>
<summary>💡 راه‌حل (برای بازشدن کلیک کن)</summary>

```python
print("FINAL:", run_agent("What is 15% of 240?"))
```

مدل `calculator("15% of 240")` می‌نویسد که پایتون معتبر نیست، پس ابزار خطای نحوی برمی‌گرداند. بعد همان فراخوانی خراب را تا سقف گام‌ها تکرار می‌کند. برنامه‌ریزی درست است، قالب‌بندی غلط، و مدل از پیام خطا چیزی یاد نمی‌گیرد. راه‌حل‌ها به ترتیب زحمت: توصیف دقیق‌تر نحو در پیام سیستمی، افزودن مثالی از درصد، برگرداندن خطا همراه راهنمایی اصلاح، یا استفاده از مدل بزرگ‌تر. مهندسی عامل در عمل از همین جنس است.

</details>

</div>

<div dir="rtl" style="text-align:right">

### ✏️ تمرین ۱۱
ابزار دومی اضافه کن. تابع `word_count(text)` بنویس، در پیام سیستمی کنار ماشین‌حساب توصیفش کن، تجزیه‌کننده را طوری گسترش بده که بشناسدش، و پرسشی بپرس که به آن نیاز دارد.

</div>

In [ ]:
# ✏️ your second tool



<div dir="rtl" style="text-align:right">

<details>
<summary>💡 راه‌حل (برای بازشدن کلیک کن)</summary>

```python
TWO_TOOL_PROMPT = TOOL_PROMPT.replace(
    'calculator(expression) - evaluates a Python arithmetic expression',
    'calculator(expression) - evaluates a Python arithmetic expression\n'
    'word_count(text) - counts the words in a piece of text')

def run_two_tool_agent(question, max_steps=4):
    messages = [{"role": "system", "content": TWO_TOOL_PROMPT},
                {"role": "user", "content": question}]
    for _ in range(max_steps):
        reply = step(messages)
        print("  model:", reply)
        call = re.search(r'ACTION:\s*(calculator|word_count)\("([^"]+)"\)', reply)
        if not call:
            answer = re.search(r"ANSWER:\s*(.+)", reply)
            return answer.group(1).strip() if answer else reply
        name, argument = call.group(1), call.group(2)
        observation = calculator(argument) if name == "calculator" else len(argument.split())
        print("  tool :", observation)
        messages.append({"role": "assistant", "content": reply})
        messages.append({"role": "user", "content": f"Observation: {observation}"})
    return None

print(run_two_tool_agent('How many words are in "the quick brown fox jumps over the lazy dog"?'))
```

با دو ابزار، مدل باید *کدام* ابزار را هم انتخاب کند و مدل ۰.۵ میلیاردی گاهی اشتباه انتخاب می‌کند. انتخاب ابزار اولین چیزی است که با بلندشدن فهرست ابزارها افت می‌کند، و برای همین عامل‌های صنعتی ابزارها را دسته‌بندی می‌کنند و به‌جای فهرست‌کردن همه، مرتبط‌ها را بازیابی می‌کنند.

</details>

</div>

<div dir="rtl" style="text-align:right">

### ✏️ تمرین ۱۲
مثال کامل‌شده را از `TOOL_PROMPT` بردار (چهار خط آخر) و پرسش ضرب را دوباره اجرا کن. مدل هر چند بار هنوز فراخوانی معتبر تولید می‌کند؟

</div>

In [ ]:
# ✏️ prompt ablation



<div dir="rtl" style="text-align:right">

<details>
<summary>💡 راه‌حل (برای بازشدن کلیک کن)</summary>

```python
SHORT_PROMPT = TOOL_PROMPT.split("Example:")[0].strip()
messages = [{"role": "system", "content": SHORT_PROMPT},
            {"role": "user", "content": "What is 4321 * 1234?"}]
print(step(messages))
```

بدون مثال، مدل معمولا اصلا ابزار را صدا نمی‌زند و از حافظه خودش جواب می‌دهد، که غلط است. مثال، تفاوت بین یک عامل کارآمد و یک عامل خراب است، و این قوی‌ترین نمایش ممکن از این نکته است که برای مدل‌های کوچک، نشان‌دادن قالب از توصیف‌کردنش بهتر جواب می‌دهد.

</details>

</div>

<div dir="rtl" style="text-align:right">

---
# بخش ۵: مقیاس، مراحل، و پایان دوره

هر آن‌چه مدل بخش ۱ تو را از یک دستیار مرزی جدا می‌کند، در یک جدول جا می‌شود:

| | ‏GPT کوچک تو | Qwen2.5-0.5B | مدل‌های مرزی |
|---|---|---|---|
| پارامترها | ۸۲۶ هزار | ۴۹۴ میلیون | صدها میلیارد |
| متن آموزش | ۱ مگابایت | ترابایت‌ها | ترابایت‌ها |
| هزینه آموزش | چند دقیقه روی لپ‌تاپ | هزاران ساعت GPU | ده‌ها میلیون دلار |
| معماری | ۴ لایه ترنسفورمر | ۲۴ لایه ترنسفورمر | ده‌ها تا صدها |
| تابع هدف | پیش‌بینی توکن بعدی | پیش‌بینی توکن بعدی | پیش‌بینی توکن بعدی |

سطر آخر هرگز عوض نمی‌شود. حرف این جلسه همین است.

**سه مرحله ساخت یک دستیار.** پیش‌آموزش همان کاری است که در بخش ۱ کردی، در مقیاس بزرگ: پیش‌بینی توکن بعدی روی متن خام، که دانش از همان‌جا می‌آید. آموزش دستوری، آموزش را روی زوج‌های گزینش‌شده دستور و پاسخ ادامه می‌دهد و همین است که یک ادامه‌دهنده متن را به چیزی تبدیل می‌کند که به پرسش جواب می‌دهد. یادگیری تقویتی از بازخورد انسانی بعد پاسخ‌های نامزد را با ترجیح انسان رتبه می‌دهد و وزن‌ها را به سمت پاسخ‌های ترجیح‌داده‌شده هل می‌دهد. به شکل این مرحله آخر نگاه کن: تولید کن، نمره بگیر، اصلاح کن، تکرار. همان حلقه حدس و خطا و اصلاح جلسه ۲ است، با ترجیح انسان در نقش سیگنال خطا.

**آن‌چه هیچ‌کدام عوض نمی‌کنند.** مدل در قلبش پیش‌بینی‌کننده توکن بعدی می‌ماند. توهم می‌زند، کمتر از مدل ۰.۵ میلیاردی اما هرگز صفر، و برای همین بخش ۳ وجود دارد. حساب قابل‌اتکا بلد نیست، و برای همین بخش ۴ وجود دارد. سوگیری‌های متن آموزشش را به ارث می‌برد، همان‌طور که جلسه ۱ هشدار داد و شکاف فارسی در بخش ۲ نشانش داد. و روانی گواه درستی نیست، که مفیدترین چیزی است که موقع استفاده از این سیستم‌ها باید به یاد داشت.

</div>

<div dir="rtl" style="text-align:right">

### آیا واقعا می‌فهمد؟

این پرسش داغ روزگار ماست، و بعد از این جلسه در جایگاهی هستی که نظری آگاهانه داشته باشی، نه نظری عاریتی.

یک طرف مدل زبانی را طوطی آماری می‌نامد: الگوهای موجود در متن را بازتولید می‌کند، نه تجربه‌ای از جهان دارد نه نیتی، و خطاهای عجیبش دقیقا شکل نفهمیدن است. دو نمونه از همین خطاها جلوی چشمت است: حاصل‌ضرب غلط و مطمئن در بخش ۲، و مدلی که در تمرین ۱۰ فراخوانی خراب را تکرار می‌کند در حالی که پیام خطا درست پیش رویش نشسته.

طرف دیگر استدلال می‌کند که پیش‌بینی به‌اندازه کافی خوب توکن بعدی، در مقیاس کافی، مدل را ناچار می‌کند ساختاری درونی بسازد که نام فهم را حق دارد، و به توانایی‌هایی اشاره می‌کند که کسی برایشان آموزش نداده، مثل همان یادگیری در بافت که در بخش ۲.۴ به کار بردی و با بزرگ‌شدن مدل‌ها خودبه‌خود ظاهر شد.

پاسخ صادقانه این است که پرسش هنوز حل‌نشده است و با اندازه‌گیری حل خواهد شد، نه با شهود این طرف یا آن طرف. آن‌چه تردیدی در آن نیست همان چیزی است که ساختی: یک پیش‌بینی‌کننده توکن بعدی، چهار لایه، توجه، گرادیان کاهشی. این‌که همین ماشین‌آلات، یک‌میلیون‌برابر بزرگ‌تر، به فهم می‌رسد یا نه، پرسشی است که این رشته هنوز جوابش را نداده، و حالا مجهزی که بحث را دنبال کنی.

</div>

<div dir="rtl" style="text-align:right">

### ✏️ تمرین ۱۳: معمای پایانی
این دنباله را ادامه بده: **۱، ۱۱، ۲۱، ۱۲۱۱، ۱۱۱۲۲۱، ؟**

هر جمله، جمله قبلی را بلندبلند توصیف می‌کند. اول خودت حلش کن، بعد از دستیار بپرس و مقایسه کن. مدل کجا اشتباه می‌کند و آیا گذاشتن قاعده در پرامپت نجاتش می‌دهد؟

</div>

In [ ]:
# ✏️ the sequence puzzle



<div dir="rtl" style="text-align:right">

<details>
<summary>💡 راه‌حل (برای بازشدن کلیک کن)</summary>

```python
puzzle = "Continue the sequence and explain the rule: 1, 11, 21, 1211, 111221, ?"
print("no hint  :", chat(puzzle, max_new_tokens=60, do_sample=False))
print("with hint:", chat(puzzle + " Hint: each term reads the previous term out loud.",
                         max_new_tokens=60, do_sample=False))
```

جواب **۳۱۲۲۱۱** است، چون جمله قبلی یعنی ۱۱۱۲۲۱ بلندبلند خوانده می‌شود «سه تا یک، دو تا دو، یک تا یک». مدل ۰.۵ میلیاردی عملا هرگز به آن نمی‌رسد، چون این قدم شمردن و گروه‌بندی می‌خواهد نه کامل‌کردن الگو، و راهنمایی فقط گاهی کمک می‌کند. مدل‌های بزرگ‌تر حلش می‌کنند و بسیاری از همان‌ها هنوز جمله بعدی‌اش را غلط می‌دهند.

تمرین آخر برازنده‌ای است، چون شبیه زبان به نظر می‌رسد و در واقع حساب است، و همین درز دقیقا همان چیزی است که این جلسه تمام مدت رویش انگشت گذاشت.

</details>

</div>

<div dir="rtl" style="text-align:right">

### ده جلسه، در یک نخ

| جلسه‌ها | چه ساختی | ایده تازه |
|---|---|---|
| ۱ و ۲ | هوش مصنوعی چیست و چطور یاد می‌گیرد | داده به‌جای قاعده |
| ۳ و ۴ | ویژگی، برچسب، خط برازش‌شده | تابع هزینه و گرادیان کاهشی |
| ۵ و ۶ | درخت، جنگل، بوستینگ، اعتبارسنجی | تعمیم و سنجش صادقانه |
| ۷ | شبکه عصبی از صفر | پس‌انتشار |
| ۸ | شبکه‌های پیچشی | معماری هم‌ساخت با داده |
| ۹ | تعبیه و لایه ترنسفورمر | توجه |
| ۱۰ | یک GPT، یک دستیار، بازیابی، یک عامل | مقیاس |

از برازش یک خط به قیمت خانه‌های تهران تا اجرای عاملی که ابزار صدا می‌زند، ماشین‌آلات هرگز عوض نشد: یک حدس، یک خطا، یک گرادیان، یک گام. آن‌چه عوض شد معماری دورش و مقیاس زیرش بود. حالا تک‌تک حلقه‌های این زنجیر چیزی است که خودت پیاده کرده‌ای، آموزش داده‌ای و بازرسی کرده‌ای.

**ادامه مسیر.** [دوره Hugging Face](https://huggingface.co/learn) برای تمرین امروزی، [fast.ai](https://course.fast.ai) برای عمق کاربردی، درس‌های CS231n و CS224n استنفورد برای نظریه، و Kaggle برای عادتی که این دوره سعی کرد بسازد: اندازه بگیر، به خطاها نگاه کن، بهتر کن، تکرار کن.

و مفیدترین کاری که به فارسی می‌شود کرد: ساختن مجموعه‌داده و محک و مدل برای زبانی که بسیار کمتر از استحقاقش از این‌ها دارد. شکافش را خودت در بخش ۲ دیدی.

</div>

<div dir="rtl" style="text-align:right">

### ✏️ تمرین ۱۴: تکلیف پایانی
۱. مدل بخش ۱ را روی متنی دیگر آموزش بده: یک دیوان شعر فارسی، کتابی از [پروژه گوتنبرگ](https://www.gutenberg.org)، یا نوشته‌های خودت. همان کد، داده متفاوت، و نمونه‌ها لحن منبع را می‌گیرند.

۲. به عامل ابزار سومی بده که واقعا به کارت بیاید، مثلا جست‌وجو در سندهای بخش ۳، تا در یک حلقه هم بازیابی کند و هم حساب.

۳. یک ادعای واقعی که این هفته یک دستیار به تو گفته بردار و با یک منبع دست‌اول راستی‌آزمایی کن. این را عادت کن، نه تمرین.

دوره همان‌جا تمام می‌شود که شروع شد، در نقطه مقابل جادو: هیچ‌چیز این سیستم‌ها بیرون از فهم نیست، و حالا هر جزئشان چیزی است که خودت ساخته‌ای. 🎓

</div>